In [8]:
import math
import re
from collections import Counter
from urllib.parse import urlparse

import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

In [9]:
df = pd.read_csv("Cleaned_ISCX_URL2016.csv")
print("Dataset shape:", df.shape)
print(df["type"].value_counts())
df.head()

Dataset shape: (641125, 2)
type
benign        428080
defacement     95308
phishing       94092
malware        23645
Name: count, dtype: int64


,url,type
0,br-icloud.com.br,phishing
1,mp3raid.com/music/krizz_kaliko.html,benign
2,bopsecrets.org/rexroth/cr/1.htm,benign
3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,http://adventure-nicaragua.net/index.php?optio...,defacement


In [10]:
class URLFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        # Match IPv4 addresses used instead of domain names
        self.ip_pattern = re.compile(
            r"^(?:http[s]?://)?(?:www\.)?"
            r"(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}"
            r"(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)"
            r"(?:[:/]|$)"
        )
        # Special characters commonly inspected in phishing and malicious URLs
        self.special_chars = ["@", "?", "-", "=", ".", "_", "/", "%", "&"]

    @staticmethod
    def _shannon_entropy(text: str) -> float:
        # Compute Shannon entropy to measure character randomness
        if not text:
            return 0.0
        counts = Counter(text)
        length = len(text)
        return -sum((count / length) * math.log2(count / length) for count in counts.values())

    def _extract_single_url(self, raw_url: str) -> dict:
        u = str(raw_url).strip()
        has_https = 1 if u.lower().startswith("https://") else 0
        has_http = 1 if u.lower().startswith("http://") else 0

        # Prepend http:// temporarily when scheme is missing so urlparse extracts netloc properly
        parse_target = u if re.match(r"^[a-zA-Z][a-zA-Z0-9+.-]*://", u) else "http://" + u
        try:
            parsed = urlparse(parse_target)
            hostname = parsed.netloc.lower().split(":")[0]
            path = parsed.path
        except ValueError:
            # Fallback for malformed bracketed hosts that raise ValueError in urlparse
            without_scheme = re.sub(r"^[a-zA-Z][a-zA-Z0-9+.-]*://", "", u)
            parts_slash = without_scheme.split("/", 1)
            hostname = parts_slash[0].lower().split(":")[0]
            path = "/" + parts_slash[1] if len(parts_slash) > 1 else ""

        has_ip = 1 if self.ip_pattern.match(u) else 0

        # Extract TLD and subdomain count for non-IP hostnames
        if has_ip or not hostname:
            tld = "ip_or_none"
            subdomain_count = 0
        else:
            parts = [p for p in hostname.split(".") if p]
            tld = parts[-1] if len(parts) > 1 else "unknown"
            subdomain_count = max(0, len(parts) - 2)

        digit_count = sum(c.isdigit() for c in u)
        letter_count = sum(c.isalpha() for c in u)

        features = {
            "url_length": len(u),
            "domain_length": len(hostname),
            "path_length": len(path),
            "url_entropy": self._shannon_entropy(u),
            "domain_entropy": self._shannon_entropy(hostname),
            "has_ip": has_ip,
            "has_https": has_https,
            "has_http": has_http,
            "digit_count": digit_count,
            "letter_count": letter_count,
            "digit_ratio": digit_count / max(1, len(u)),
            "subdomain_count": subdomain_count,
            "tld": tld,
        }

        # Use ASCII codes in column names to keep feature names clean and consistent
        for ch in self.special_chars:
            features[f"count_{ord(ch)}"] = u.count(ch)

        return features

    def fit(self, X, y=None):
        # Stateless feature extractor
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            urls = X.iloc[:, 0].astype(str)
        elif isinstance(X, pd.Series):
            urls = X.astype(str)
        else:
            urls = pd.Series(np.ravel(X)).astype(str)

        rows = [self._extract_single_url(u) for u in urls]
        return pd.DataFrame(rows, index=urls.index)

In [11]:
# Drop rows with missing targets or URLs to avoid NaN errors during splitting
df_cleaned = df.dropna(subset=["url", "type"])

X = df_cleaned[["url"]]
y = df_cleaned["type"]

# Stratified train/test split before any fitting or scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)

Train shape: (512900, 1) | Test shape: (128225, 1)


In [12]:
numeric_cols = [
    "url_length",
    "domain_length",
    "path_length",
    "url_entropy",
    "domain_entropy",
    "has_ip",
    "has_https",
    "has_http",
    "digit_count",
    "letter_count",
    "digit_ratio",
    "subdomain_count",
    "count_64",
    "count_63",
    "count_45",
    "count_61",
    "count_46",
    "count_95",
    "count_47",
    "count_37",
    "count_38",
]
categorical_cols = ["tld"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", max_categories=30, sparse_output=False), categorical_cols),
    ]
)

preprocessing_pipeline = Pipeline(
    steps=[
        ("feature_extraction", URLFeatureExtractor()),
        ("preprocessing", preprocessor),
    ]
)

# Fit strictly on X_train; transform X_test without fitting on it
X_train_processed = preprocessing_pipeline.fit_transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)
feature_names = preprocessing_pipeline.named_steps["preprocessing"].get_feature_names_out()

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape:", X_test_processed.shape)

Processed X_train shape: (512900, 51)
Processed X_test shape: (128225, 51)


In [13]:
train_raw = preprocessing_pipeline.named_steps["feature_extraction"].transform(X_train)
full_raw = preprocessing_pipeline.named_steps["feature_extraction"].transform(X)
scaler = preprocessing_pipeline.named_steps["preprocessing"].named_transformers_["num"]

diff_train = np.max(np.abs(scaler.mean_ - train_raw[numeric_cols].mean().values))
diff_full = np.max(np.abs(scaler.mean_ - full_raw[numeric_cols].mean().values))

print(f"Max diff vs X_train mean: {diff_train:.2e}")
print(f"Max diff vs full dataset mean: {diff_full:.6f}")

assert diff_train < 1e-12, "Data leakage detected: scaler mean does not match X_train"
assert diff_full > 1e-5, "Unexpected match between scaler mean and full dataset"
print("Leakage check passed.")

Max diff vs X_train mean: 0.00e+00
Max diff vs full dataset mean: 0.038600
Leakage check passed.


In [14]:
joblib.dump(preprocessing_pipeline, "url_preprocessing_pipeline.joblib")
np.savez_compressed(
    "processed_url_splits.npz",
    X_train=X_train_processed,
    X_test=X_test_processed,
    y_train=y_train.to_numpy(),
    y_test=y_test.to_numpy(),
    feature_names=feature_names,
)
print("Saved url_preprocessing_pipeline.joblib and processed_url_splits.npz")

Saved url_preprocessing_pipeline.joblib and processed_url_splits.npz
